In [3]:
"""
Stage 3 - Cosine Similarity Graph and Louvain Community Detection
Uses python-igraph instead of NetworkX for memory-efficient graph storage.
igraph stores graph data in C structures, handling millions of edges safely.
Algorithm: Louvain community detection
"""

import numpy as np
import pandas as pd
import igraph as ig
import json
import time
from pathlib import Path
from collections import Counter

#  Paths and constants
EMB_DIR    = Path("../embeddings")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDINGS    = EMB_DIR / "embeddings.npy"
SAMPLE_CSV    = EMB_DIR / "stratified_sample.csv"
RESULTS_JSON  = OUTPUT_DIR / "stage3_results.json"
COMMUNITY_CSV = OUTPUT_DIR / "community_assignments.csv"

#  Parameters 
DEFAULT_THRESHOLD  = 0.95
SENSITIVITY_VALUES = [0.85, 0.90, 0.95]
LOUVAIN_SEED       = 42
LOUVAIN_RESOLUTION = 1.0
HOMOGENEITY_TARGET = 0.70
CHUNK_SIZE         = 500

#  Load 
def load_data():
    print("Loading embeddings and sample metadata...")
    embeddings = np.load(EMBEDDINGS)
    sample     = pd.read_csv(
        SAMPLE_CSV, index_col="sample_id", low_memory=False
    )
    print(f"  Embeddings : {embeddings.shape}")
    print(f"  Sample rows: {len(sample):,}")
    assert len(embeddings) == len(sample)
    return embeddings, sample

# Build edge list in chunks 
def build_edge_list(embeddings, threshold):
    """
    Returns edges as two numpy arrays (sources, targets) and weights array.
    Never builds full N×N matrix. Peak memory ~20 MB per chunk.
    """
    n          = len(embeddings)
    n_chunks   = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
    src_list, dst_list, wgt_list = [], [], []

    print(f"  Building edge list — {n_chunks} chunks...")

    for chunk_idx in range(n_chunks):
        start_i = chunk_idx * CHUNK_SIZE
        end_i   = min(start_i + CHUNK_SIZE, n)

        # (chunk_size, n) similarity sub-matrix
        chunk_sim = np.dot(
            embeddings[start_i:end_i],
            embeddings.T
        ).astype(np.float32)

        for local_i, global_i in enumerate(range(start_i, end_i)):
            # Only upper triangle: j > global_i
            row   = chunk_sim[local_i, global_i + 1:]
            above = np.where(row >= threshold)[0]
            if len(above) > 0:
                src_list.append(
                    np.full(len(above), global_i, dtype=np.int32)
                )
                dst_list.append((global_i + 1 + above).astype(np.int32))
                wgt_list.append(row[above].astype(np.float32))

        del chunk_sim

        if (chunk_idx + 1) % 5 == 0 or chunk_idx == n_chunks - 1:
            total_edges = sum(len(x) for x in src_list)
            print(f"    Chunk {chunk_idx+1:3d}/{n_chunks} | "
                  f"edges so far: {total_edges:,}")

    if not src_list:
        return np.array([], dtype=np.int32), \
               np.array([], dtype=np.int32), \
               np.array([], dtype=np.float32)

    sources = np.concatenate(src_list)
    targets = np.concatenate(dst_list)
    weights = np.concatenate(wgt_list)
    return sources, targets, weights

# Build igraph and run Louvain 
def build_and_cluster(sources, targets, weights, sample, threshold):
    """
    Build igraph graph and run Louvain community detection.
    igraph stores everything in C memory — handles millions of edges safely.
    """
    n       = len(sample)
    n_edges = len(sources)

    if n_edges == 0:
        print(f"  No edges at threshold {threshold}")
        return None, None

    print(f"  Building igraph with {n:,} nodes and {n_edges:,} edges...")
    start = time.time()

    G = ig.Graph(n=n, directed=False)
    G.add_edges(list(zip(sources.tolist(), targets.tolist())))
    G.es["weight"] = weights.tolist()

    # Add node attributes
    G.vs["tactic"]       = sample["attck_tactic"].tolist()
    G.vs["technique_id"] = sample["attck_technique_id"].tolist()
    G.vs["label"]        = sample["Label"].tolist()
    G.vs["day"]          = sample["day"].tolist()

    elapsed = time.time() - start
    print(f"  igraph built in {elapsed:.1f}s")

    # Run Louvain
    print(f"  Running Louvain (seed={LOUVAIN_SEED}, resolution={LOUVAIN_RESOLUTION})...")
    start     = time.time()
    partition = G.community_multilevel(
    weights    = "weight",
    resolution = LOUVAIN_RESOLUTION,
)
    elapsed = time.time() - start
    print(f"  Louvain complete in {elapsed:.1f}s — {len(partition)} communities")

    # Build membership dict: node_id -> community_id
    membership = {}
    for comm_id, members in enumerate(partition):
        for node in members:
            membership[node] = comm_id

    # Compute homogeneity per community
    communities       = {}
    for node, cid in membership.items():
        communities.setdefault(cid, []).append(node)

    homogeneity_scores = []
    community_details  = []

    for cid, nodes in communities.items():
        tactics  = [G.vs[nd]["tactic"] for nd in nodes]
        counter  = Counter(tactics)
        dominant = counter.most_common(1)[0][0]
        dom_n    = counter.most_common(1)[0][1]
        hom      = dom_n / len(nodes)
        homogeneity_scores.append(hom)
        community_details.append({
            "community_id":    cid,
            "size":            len(nodes),
            "dominant_tactic": dominant,
            "homogeneity":     round(hom, 4),
            "tactic_mix":      dict(counter),
            "coherent":        hom >= HOMOGENEITY_TARGET,
        })

    arr      = np.array(homogeneity_scores)
    sizes    = [len(v) for v in communities.values()]
    coherent = sum(1 for h in homogeneity_scores if h >= HOMOGENEITY_TARGET)

    result = {
        "threshold":              threshold,
        "n_nodes":                n,
        "n_edges":                n_edges,
        "n_communities":          len(communities),
        "mean_homogeneity":       round(float(arr.mean()), 4),
        "median_homogeneity":     round(float(np.median(arr)), 4),
        "min_homogeneity":        round(float(arr.min()), 4),
        "max_homogeneity":        round(float(arr.max()), 4),
        "pct_coherent":           round(coherent / len(communities) * 100, 1),
        "n_coherent_communities": coherent,
        "mean_community_size":    round(float(np.mean(sizes)), 1),
        "median_community_size":  int(np.median(sizes)),
        "max_community_size":     int(np.max(sizes)),
        "min_community_size":     int(np.min(sizes)),
        "community_details":      community_details,
    }
    return result, membership

# Sensitivity analysis 
def sensitivity_analysis(embeddings, sample):
    print("\n")

    all_results      = {}
    best_membership  = None
    best_result      = None

    for t in SENSITIVITY_VALUES:
        print(f"\n{'─'*50}")
        print(f"  Threshold t = {t}")
        print(f"{'─'*50}")
        start = time.time()

        sources, targets, weights = build_edge_list(embeddings, t)
        result, membership        = build_and_cluster(
            sources, targets, weights, sample, t
        )
        elapsed = time.time() - start

        if result is None:
            continue

        all_results[str(t)] = result
        print(f"  Done in {elapsed:.1f}s | "
              f"communities={result['n_communities']} | "
              f"mean_hom={result['mean_homogeneity']:.4f} | "
              f"coherent={result['pct_coherent']}%")

        if t == DEFAULT_THRESHOLD:
            best_membership = membership
            best_result     = result

        # Free edge arrays before next threshold
        del sources, targets, weights

    return all_results, best_membership, best_result

# Save community assignments 
def save_community_assignments(sample, best_membership, best_result):
    sample_out = sample.copy()
    sample_out["community_id"] = [
        best_membership.get(i, -1) for i in range(len(sample))
    ]
    comm_lookup = {
        c["community_id"]: c for c in best_result["community_details"]
    }
    sample_out["community_dominant_tactic"] = sample_out["community_id"].map(
        lambda c: comm_lookup.get(c, {}).get("dominant_tactic", "Unknown")
    )
    sample_out["community_homogeneity"] = sample_out["community_id"].map(
        lambda c: comm_lookup.get(c, {}).get("homogeneity", 0.0)
    )
    sample_out["community_coherent"] = sample_out["community_id"].map(
        lambda c: comm_lookup.get(c, {}).get("coherent", False)
    )
    sample_out["community_size"] = sample_out["community_id"].map(
        lambda c: comm_lookup.get(c, {}).get("size", 0)
    )
    sample_out.to_csv(COMMUNITY_CSV)
    print(f"\n  Saved to: {COMMUNITY_CSV}")
    return sample_out

#  Main 
def main():
    embeddings, sample = load_data()

    all_results, best_membership, best_result = sensitivity_analysis(
        embeddings, sample
    )

    if best_result is None:
        print("ERROR: No results at default threshold.")
        return

    # Save summary JSON
    summary = {
        t: {k: v for k, v in r.items() if k != "community_details"}
        for t, r in all_results.items()
    }
    with open(RESULTS_JSON, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n  Results saved to: {RESULTS_JSON}")

    sample_out = save_community_assignments(
        sample, best_membership, best_result
    )

    #report 
    r = best_result
    print(f"  Alerts              : {r['n_nodes']:,}")
    print(f"  Edges               : {r['n_edges']:,}")
    print(f"  Communities         : {r['n_communities']}")
    print(f"  Mean homogeneity    : {r['mean_homogeneity']:.4f}")
    print(f"  Median homogeneity  : {r['median_homogeneity']:.4f}")
    print(f"  Coherent (>={HOMOGENEITY_TARGET})  : "
          f"{r['n_coherent_communities']} ({r['pct_coherent']}%)")
    print(f"  Largest community   : {r['max_community_size']} alerts")
    print(f"  Smallest community  : {r['min_community_size']} alerts")

    print(f"\n  Top 10 communities by size:")
    top10 = sorted(
        r["community_details"], key=lambda x: x["size"], reverse=True
    )[:10]
    print(f"  {'ID':>5} {'Size':>6} {'Hom':>8} "
          f"{'Coherent':>9}  Dominant Tactic")
    print(f"  {'-'*55}")
    for c in top10:
        print(f"  {c['community_id']:>5} "
              f"{c['size']:>6} "
              f"{c['homogeneity']:>8.4f} "
              f"{'YES' if c['coherent'] else 'NO':>9}  "
              f"{c['dominant_tactic']}")

    print(f"\n  Sensitivity summary:")
    print(f"  {'Threshold':>10} {'Edges':>12} {'Communities':>13} "
          f"{'Mean Hom':>10} {'% Coherent':>12}")
    print(f"  {'-'*60}")
    for t, res in sorted(all_results.items()):
        print(f"  {float(t):>10.2f} "
              f"{res['n_edges']:>12,} "
              f"{res['n_communities']:>13} "
              f"{res['mean_homogeneity']:>10.4f} "
              f"{res['pct_coherent']:>11.1f}%")


if __name__ == "__main__":
    main()

Loading embeddings and sample metadata...
  Embeddings : (10673, 384)
  Sample rows: 10,673



──────────────────────────────────────────────────
  Threshold t = 0.85
──────────────────────────────────────────────────
  Building edge list — 22 chunks...
    Chunk   5/22 | edges so far: 6,070,037
    Chunk  10/22 | edges so far: 10,635,563
    Chunk  15/22 | edges so far: 13,535,365
    Chunk  20/22 | edges so far: 14,798,860
    Chunk  22/22 | edges so far: 14,857,979
  Building igraph with 10,673 nodes and 14,857,979 edges...
  igraph built in 7.2s
  Running Louvain (seed=42, resolution=1.0)...
  Louvain complete in 6.6s — 5 communities
  Done in 15.1s | communities=5 | mean_hom=0.8010 | coherent=60.0%

──────────────────────────────────────────────────
  Threshold t = 0.9
──────────────────────────────────────────────────
  Building edge list — 22 chunks...
    Chunk   5/22 | edges so far: 3,666,950
    Chunk  10/22 | edges so far: 6,448,408
    Chunk  15/22 | edges so far: 8,220,551